In [2]:
# Memanggil library

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

from scipy.sparse import hstack

In [ ]:
# Memanggil data

train = pd.read_csv("training.csv")
predict = pd.read_csv("predict.csv")
submission_template = pd.read_csv("submission-template.csv")

print("TRAIN :", train.shape)
print("PREDICT:", predict.shape)
print("SUBMISSION:", submission_template.shape)


In [4]:
# Menyiapkan data

X = train.drop(columns=["category"])
y = train["category"]

SEED = 42

In [17]:
# Membagi data

label_counts = y.value_counts()

common_mask = y.map(label_counts) >= 2
rare_mask = y.map(label_counts) < 2

X_common = X[common_mask]
y_common = y[common_mask]

X_rare = X[rare_mask]
y_rare = y[rare_mask]

X_train, X_val, y_train, y_val = train_test_split(
    X_common,
    y_common,
    test_size=0.20,
    stratify=y_common,
    random_state=SEED
)

X_train = pd.concat([X_train, X_rare])
y_train = pd.concat([y_train, y_rare])

In [16]:
# Menyiapkan fitur teks

text_columns = [
    "url",
    "brand",
    "domain",
    "sld",
    "registrar"
]

def combine_text(df):
    return (
        df[text_columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )

X_train_text = combine_text(X_train)
X_val_text = combine_text(X_val)

In [15]:
# Membuat Word TF-IDF

word_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1
)

X_train_word = word_vectorizer.fit_transform(X_train_text)
X_val_word = word_vectorizer.transform(X_val_text)

In [14]:
# Membuat Character TF-IDF

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2
)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_val_char = char_vectorizer.transform(X_val_text)

In [13]:
# Membuat fitur confidence

conf_encoder = OneHotEncoder(handle_unknown="ignore")

conf_train = conf_encoder.fit_transform(
    X_train[["confidence_level"]]
)

conf_val = conf_encoder.transform(
    X_val[["confidence_level"]]
)

In [12]:
# Membuat fitur waktu

def make_time_features(df):
    dt = pd.to_datetime(
        df["discovered"],
        errors="coerce"
    )

    return pd.DataFrame({
        "year": dt.dt.year.fillna(-1).astype(str),
        "month": dt.dt.month.fillna(-1).astype(str),
        "dayofweek": dt.dt.dayofweek.fillna(-1).astype(str),
        "hour": dt.dt.hour.fillna(-1).astype(str)
    })

time_train = make_time_features(X_train)
time_val = make_time_features(X_val)

time_encoder = OneHotEncoder(handle_unknown="ignore")

time_train_encoded = time_encoder.fit_transform(time_train)
time_val_encoded = time_encoder.transform(time_val)

In [ ]:
# Menggabungkan fitur

X_train_final = hstack([
    X_train_word,
    X_train_char,
    conf_train,
    time_train_encoded
])

X_val_final = hstack([
    X_val_word,
    X_val_char,
    conf_val,
    time_val_encoded
])

In [19]:
# Evaluasi

pred_val = final_model.predict(X_val_final)

macro_f1 = f1_score(
    y_val,
    pred_val,
    average="macro"
)

print("Macro-F1:", macro_f1)

Macro-F1: 0.7448463291701813


In [20]:
# Mencari C terbaik

for c in [0.25, 0.5, 1.0, 2.0, 3.0, 5.0]:
    model = LinearSVC(
        C=c,
        random_state=SEED,
        max_iter=5000
    )

    model.fit(X_train_final, y_train)

    pred = model.predict(X_val_final)

    score = f1_score(
        y_val,
        pred,
        average="macro"
    )

    print(f"C={c} -> Macro-F1={score:.4f}")

C=0.25 -> Macro-F1=0.6548
C=0.5 -> Macro-F1=0.6401
C=1.0 -> Macro-F1=0.7448
C=2.0 -> Macro-F1=0.7451
C=3.0 -> Macro-F1=0.7441
C=5.0 -> Macro-F1=0.7444


In [21]:
# Mencari konfigurasi Character TF-IDF

for ngram in [(2, 5), (3, 6), (3, 7), (4, 6), (4, 7)]:
    char_test = TfidfVectorizer(
        analyzer="char",
        ngram_range=ngram,
        min_df=2
    )

    char_train = char_test.fit_transform(X_train_text)
    char_val = char_test.transform(X_val_text)

    X_train_test = hstack([
        X_train_word,
        char_train,
        conf_train,
        time_train_encoded
    ])

    X_val_test = hstack([
        X_val_word,
        char_val,
        conf_val,
        time_val_encoded
    ])

    model = LinearSVC(
        C=2.0,
        random_state=SEED,
        max_iter=5000
    )

    model.fit(X_train_test, y_train)

    pred = model.predict(X_val_test)

    score = f1_score(
        y_val,
        pred,
        average="macro"
    )

    print(f"ngram={ngram} -> Macro-F1={score:.4f}")

ngram=(2, 5) -> Macro-F1=0.7453
ngram=(3, 6) -> Macro-F1=0.7465
ngram=(3, 7) -> Macro-F1=0.7454
ngram=(4, 6) -> Macro-F1=0.7454
ngram=(4, 7) -> Macro-F1=0.7454


In [22]:
# Mengunci Character TF-IDF terbaik

char_best = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 6),
    min_df=2
)

char_train_best = char_best.fit_transform(X_train_text)
char_val_best = char_best.transform(X_val_text)

In [23]:
# Mencari konfigurasi Word TF-IDF

for ngram in [(1, 1), (1, 2), (1, 3), (2, 3)]:
    word_test = TfidfVectorizer(
        lowercase=True,
        ngram_range=ngram,
        min_df=1
    )

    word_train = word_test.fit_transform(X_train_text)
    word_val = word_test.transform(X_val_text)

    X_train_test = hstack([
        word_train,
        char_train_best,
        conf_train,
        time_train_encoded
    ])

    X_val_test = hstack([
        word_val,
        char_val_best,
        conf_val,
        time_val_encoded
    ])

    model = LinearSVC(
        C=2.0,
        random_state=SEED,
        max_iter=5000
    )

    model.fit(X_train_test, y_train)

    pred = model.predict(X_val_test)

    score = f1_score(
        y_val,
        pred,
        average="macro"
    )

    print(f"ngram={ngram} -> Macro-F1={score:.4f}")

ngram=(1, 1) -> Macro-F1=0.7454
ngram=(1, 2) -> Macro-F1=0.7465
ngram=(1, 3) -> Macro-F1=0.7455
ngram=(2, 3) -> Macro-F1=0.7455


In [24]:
# Menguji class weight

X_train_best = hstack([
    X_train_word,
    char_train_best,
    conf_train,
    time_train_encoded
])

X_val_best = hstack([
    X_val_word,
    char_val_best,
    conf_val,
    time_val_encoded
])

for weight in [None, "balanced"]:
    model = LinearSVC(
        C=2.0,
        class_weight=weight,
        random_state=SEED,
        max_iter=5000
    )

    model.fit(X_train_best, y_train)

    pred = model.predict(X_val_best)

    score = f1_score(
        y_val,
        pred,
        average="macro"
    )

    print(f"class_weight={weight} -> Macro-F1={score:.4f}")

class_weight=None -> Macro-F1=0.7465
class_weight=balanced -> Macro-F1=0.6814


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [25]:
# Prediksi model terbaik

best_model = LinearSVC(
    C=2.0,
    class_weight=None,
    random_state=SEED,
    max_iter=10000
)

best_model.fit(
    X_train_best,
    y_train
)

pred_best = best_model.predict(X_val_best)

print(
    "Macro-F1:",
    f1_score(
        y_val,
        pred_best,
        average="macro"
    )
)

Macro-F1: 0.7465474096964587


In [45]:
# Menyiapkan seluruh data

X_full = train.drop(columns=["category"])
y_full = train["category"]

X_test = predict.drop(columns=["id"])

print("Training:", X_full.shape)
print("Test:", X_test.shape)


Training: (8400, 9)
Test: (1500, 9)


In [ ]:
# Membuat fitur teks seluruh data
X_full_text = combine_text(X_full)
X_test_text = combine_text(X_test)

word_vectorizer_full = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1
)

char_vectorizer_full = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 6),
    min_df=2
)

X_full_word = word_vectorizer_full.fit_transform(X_full_text)
X_test_word = word_vectorizer_full.transform(X_test_text)

X_full_char = char_vectorizer_full.fit_transform(X_full_text)
X_test_char = char_vectorizer_full.transform(X_test_text)

print("Word:", X_full_word.shape)
print("Char:", X_full_char.shape)


In [ ]:
# Membuat fitur confidence seluruh data
conf_encoder_full = OneHotEncoder(handle_unknown="ignore")

conf_full = conf_encoder_full.fit_transform(X_full[["confidence_level"]])
conf_test = conf_encoder_full.transform(X_test[["confidence_level"]])


In [ ]:
# Membuat fitur waktu seluruh data
time_full = make_time_features(X_full)
time_test = make_time_features(X_test)

time_encoder_full = OneHotEncoder(handle_unknown="ignore")
time_full_encoded = time_encoder_full.fit_transform(time_full)
time_test_encoded = time_encoder_full.transform(time_test)


In [ ]:
# Menggabungkan seluruh fitur
X_full_final = hstack([
    X_full_word,
    X_full_char,
    conf_full,
    time_full_encoded
])

X_test_final = hstack([
    X_test_word,
    X_test_char,
    conf_test,
    time_test_encoded
])

print("Training:", X_full_final.shape)
print("Test:", X_test_final.shape)


In [ ]:
# Training seluruh data
final_model_full = LinearSVC(
    C=2.0,
    class_weight=None,
    random_state=SEED,
    max_iter=10000
)

final_model_full.fit(X_full_final, y_full)
print("Model final selesai dilatih")


In [ ]:
# Prediksi data test
final_pred = final_model_full.predict(X_test_final)
print("Jumlah prediksi:", len(final_pred))
print("Jumlah kelas:", len(set(final_pred)))


In [ ]:
# Membuat submission
submission_final = submission_template.copy()
submission_final["category"] = final_pred

print(submission_final.head())
print("Shape:", submission_final.shape)


In [ ]:
# Menyimpan submission
submission_final.to_csv("submission.csv", index=False)
print("submission.csv berhasil dibuat")


In [ ]:
# Membuat submission

pred_map = pd.Series(
    pred_test,
    index=predict["id"]
)

submission = submission_template.copy()
submission["category"] = submission["id"].map(pred_map)

print("Jumlah baris submission:", len(submission))
print("Category kosong:", submission["category"].isna().sum())

submission.head()

Jumlah baris submission: 1500
Category kosong: 0


,id,category
0,PEDAS-bf8bc60352a3,phishing
1,PEDAS-7dd13e6859b0,online gambling
2,PEDAS-0fe0d926e0d2,phishing
3,PEDAS-37db191c480d,phishing
4,PEDAS-2d06b168bfcc,online gambling
